In [ ]:
import numpy as np
import tqdm
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import sys
sys.path.append('../../sedov_theory/python/')
from sedov_theory import SedovTalorProblem
import yt
from analysis_tool import CastroSimulation

In [ ]:
run_dir = '../run/'
file_start = 'plt_1d_'

In [ ]:
cs = CastroSimulation(run_dir, file_start)

# Compare data with theory

In [ ]:
# Calculate analytical solution
gamma = 5./3
rho = 1.67e-6 # g / cm^3
E = 4400 # erg / cm

sol = SedovTalorProblem(gamma, E, rho)

In [ ]:
plt.figure(figsize=(5, 4))
plt.clf()
level = 3
time = 4e-9

labels = {
    'rho_Hn': 'Density of Hydrogen atoms',
    'rho_Hp': 'Density of Hydrogen ions',
    'density': 'Total Hydrogen density',
}
colors = {
    'rho_Hn': 'gray', 
    'rho_Hp': 'blue',
    'density': 'black'
}

for quantity in ['density', 'rho_Hn', 'rho_Hp']:
    r, q, t = cs.extract_data( time, quantity, level)
    # Convert r from cm to microns
    # Convert massic density in cc to number density in cc
    plt.plot(1e4*r, q/1.67e-6, '-o', label=labels[quantity], color=colors[quantity])

# Plot Sedov-Taylor solution
r_th = r
plt.plot( 1e4*r_th, sol.evaluate( 'density', r, t )/1.67e-6, 'k--', label='Sedov-Taylor theory' )
plt.legend(loc=1)

plt.ylabel('Density (10$^{18}$ cm$^{-3}$)')
plt.xlabel('r ($\mu m$)')
plt.title(f'Density at t={t*1.e9:.1f} ns')

In [ ]:
# plot data at difference resolutions
quantity = 'pressure'

plt.clf()
for level in range(3,-1,-1):
    r, q, t = cs.extract_data( 1.e-9, quantity, level)
    plt.plot(r, q, '-o', label='level=%d'%level)
    if level == 3:
        r_th = r
        t_th = t
#plt.plot( r_th, sol.evaluate( quantity, r_th, t_th ), 'k--', label='analytical' )
plt.legend(loc=0)

In [ ]:
# plot data at difference resolutions
quantity = 'Temp'

plt.clf()
for level in range(3,-1,-1):
    r, q, t = cs.extract_data( 80, quantity, level)
    plt.plot(r, q, '-o', label='level=%d'%level)
    if level == 3:
        r_th = r
        t_th = t
#plt.plot( r_th, sol.evaluate( quantity, r_th, t_th ), 'k--', label='analytical' )
plt.legend(loc=0)

In [ ]:
# Extract data from different time 
# Note that time is not regularly spaced
q_arr = []
rmax_arr = []
for time in tqdm.tqdm( cs.output_times ):
    r, q, t = cs.extract_data(time, 'density', level=2)
    rmax = r[np.argmax(q)]
    rmax_arr.append(rmax)
    q_arr.append(q)
q_arr = np.stack(q_arr)
t_arr = cs.output_times.copy()
r_arr = r

# Interpolate on a grid with regularly-spaced time
interp = RegularGridInterpolator(points=(t_arr, r_arr), values=q_arr, bounds_error=False, fill_value=None)
t_interp, r_interp = np.meshgrid(
    np.linspace(0, t_arr.max(), 1000), 
    np.linspace(0, r_arr.max(), 1000), indexing='ij')
q_interp = interp((t_interp, r_interp))

In [ ]:
plt.figure(figsize=(6, 2))
plt.imshow(q_interp.T/1.67e-6, origin='lower', 
           extent=[0, t_arr.max()*1.e9, 0, r_arr.max()*1.e4], 
           aspect='auto', vmax=3)
cb = plt.colorbar()
cb.set_label('H density (10$^{18}$ cm$^{-3}$)')
#plt.plot(t_arr, rmax_arr, 'r-')
r_analytical = sol.blast_radius(t_arr)
#plt.plot(t_arr*1.e9, r_analytical*1.e4, 'r--', label='Sedov-Taylor theory')
#plt.legend(loc=0)
plt.ylabel('r ($\mu m$)')
plt.xlabel('t (ns)')
plt.ylim(0,300)


# 2D SlicePlot of the blast

In [ ]:
for iteration in range(0, 200, 2):
    print(iteration)
    ds = yt.load('../run/plt_2d_%05d/' %iteration, hint="castro")
    quantity = 'density'
    sl = yt.SlicePlot(ds, 'z', quantity)
    sl.set_log(quantity, False)
    sl.set_zlim(quantity, 0, 3e-6)
    sl.annotate_grids()
    sl.save('./img%05d.png' %iteration)

In [ ]:
ds = yt.load('../run/plt_2d_00400/', hint="castro")
quantity = 'rho_Hp'
sl = yt.SlicePlot(ds, 'theta', quantity)
sl.set_log(quantity, False)
sl.annotate_grids()